In this notebook we will take output from step 4 excel file and alos add RAG performace columns in that. so we can also understand RAG.

In [1]:
import pandas as pd

#############################################################################
######### CHANGE OUTPUT FOLDER FILE HERE ####################################
#############################################################################
file_path = r"C:\Research-Paper\PAPER-WORK-2024\Outputs\Spider-Data-7.xlsx"

# Read the Excel file into a DataFrame
spider_excel_df = pd.read_excel(file_path)

# Display the first few rows of the DataFrame
spider_excel_df.head()

,db_id,spider_query,question,text2sql_query
0,farm,SELECT count(*) FROM farm,How many farms are there?,"SELECT COUNT(DISTINCT f.Farm_ID) FROM ""farm"" f;"
1,farm,SELECT count(*) FROM farm,Count the number of farms.,"SELECT COUNT(DISTINCT f.""Farm_ID"") FROM ""farm"" f;"
2,farm,SELECT Total_Horses FROM farm ORDER BY Total_H...,List the total number of horses on farms in as...,"SELECT f.""Farm_ID"", f.""Year"", f.""Total_Horses""..."
3,farm,SELECT Total_Horses FROM farm ORDER BY Total_H...,"What is the total horses record for each farm,...","SELECT f.Farm_ID, f.Total_Horses FROM farm f O..."
4,farm,SELECT Hosts FROM farm_competition WHERE Theme...,What are the hosts of competitions whose theme...,SELECT DISTINCT fc.Hosts FROM farm_competition...


In [2]:
from langchain.docstore.document import Document
import os
import re
import streamlit as st
from itertools import permutations

db_id_dict = {}
def init_document():
    db_id_dict = {}
    documents = []
    ##############################################################################
    ######### CHANGE INPUT RAG FILE FORLDER HERE #################################
    ##############################################################################
    # directory_path = 'C:\Research-Paper\PAPER-WORK-2024\RAG-FILES'
    directory_path = 'C:\Research-Paper\PAPER-WORK-2024\Spider-Data-7'
    # The directory contains documents in format <TABLE_NAME>__<DB_ID>.txt
    files = os.listdir(directory_path)

    for file in files:
        file_path = os.path.join(directory_path, file)
        if os.path.isfile(file_path):
            with open(file_path, 'r') as f:
                print("Reading file => ",file_path)
                match = re.search(r"(.*?)__(.*?)\.txt", file)
                table_name = match.group(1)
                db_id = match.group(2)
                if db_id not in db_id_dict:
                    db_id_dict[db_id] = [table_name]
                else:
                    db_id_dict[db_id].append(table_name)
                documents.append(Document(page_content=f.read(), metadata={"source": "local", "context": db_id,"table name":table_name}))
    print("Number of documents currently used: ", len(documents))
    return documents

In [3]:
from langchain.embeddings import HuggingFaceEmbeddings # Source: https://medium.com/international-school-of-ai-data-science/implementing-rag-with-langchain-and-hugging-face-28e3ea66c5f7

def init_embedding():
    print("Init embeddings...")
    # Define the path to the pre-trained model you want to use
    # This is a lightweight and fast model with good performance on semantic textual similarity tasks
    model_path = "sentence-transformers/all-MiniLM-L12-v2"

    """
        Currently the above embedding works fine, but in future we can use more embeddings from https://huggingface.co/spaces/mteb/leaderboard
        As the data grows, some of the embeddings we have tested already are below we can test again these on new data and try among these:
            1. sentence-transformers/all-MiniLM-L12-v2 (RANK: 123)
            2. Alibaba-NLP/gte-large-en-v1.5 (RANK: 19)
            3. dunzhang/stella_en_1.5B_v5 (RANK: 3)
            4. Alibaba-NLP/gte-Qwen2-1.5B-instruct (RANK: 13)
            5. intfloat/e5-base-v2 (USED IN TAG PAPER: https://arxiv.org/pdf/2408.14717)
    """

    # Create a dictionary with model configuration options, specifying to use the CPU for computations
    model_kwargs = {'device': 'cpu'}
    #model_kwargs = {'device': 'cpu', 'trust_remote_code': True} # Sometime this config works for other embeddings in list above

    # Create a dictionary with encoding options, specifically setting 'normalize_embeddings' to False
    encode_kwargs = {'normalize_embeddings': False}

    # Initialize an instance of HuggingFaceEmbeddings with the specified parameters
    embeddings = HuggingFaceEmbeddings(
        model_name=model_path,  # Provide the pre-trained model's path
        model_kwargs=model_kwargs,  # Pass the model configuration options
        encode_kwargs=encode_kwargs  # Pass the encoding options
    )
    print("Loaded embedding : ",model_path)
    return embeddings

In [4]:
from langchain.vectorstores import FAISS
# Global variable to store the cached result
cached_db = None
cached_embed = None

def init_database():
    # print("Init database...")
    global cached_db
    # print("CACHED_DB = ",cached_db)
    global cached_embed
    # print("EMBED = ",cached_embed)

    if cached_embed is None:
        print("Loading Embedding...")
        cached_embed = init_embedding()

    if True:
        # If the result is already cached, return it
        if cached_db is not None:
            print("Using cached database...")
            return cached_db
        documents = init_document()
        db = FAISS.from_documents(documents, cached_embed)
        # Cache the result
        cached_db = db
        return db

In [5]:
def similarity_k_search(question,k=3):
    db = init_database()
    searchDocs = db.similarity_search_with_score(question,k=k)
    # get a list with page content and score
    for i in range(k):
        print(searchDocs[i][0].metadata["table name"]," from db_id : ",searchDocs[i][0].metadata["context"]," scored ==>",searchDocs[i][1])
    return [(searchDocs[i][0].page_content,searchDocs[i][1],searchDocs[i][0].metadata["table name"],searchDocs[i][0].metadata["context"]) for i in range(k)]

In [6]:
question = "How many farms are there?"
answer = similarity_k_search(question,k=3)

Loading Embedding...
Init embeddings...


C:\Users\GuPr564\AppData\Local\anaconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\GuPr564\AppData\Local\anaconda3\Lib\site-packages\torchvision\datapoints\__init__.py:12: UserWarning: The torchvision.datapoints and torchvision.transforms.v2 namespaces are still Beta. While we do not expect major breaking changes, some APIs may still change according to user feedback. Please submit any feedback you may have in this issue: https://github.com/pytorch/vision/issues/6753, and you can also check out https://github.com/pytorch/vision/issues/7319 to learn more about the APIs that we suspect might involve future changes. You can silence this warning by calling torchvision.disable_beta_transforms_warning().
  warnings.warn(_BETA_TRANSFORMS_WARNING)
C:\Users\GuPr564\AppData\Local\anaconda3\Lib\site-packages\t

Loaded embedding :  sentence-transformers/all-MiniLM-L12-v2
Reading file =>  C:\Research-Paper\PAPER-WORK-2024\Spider-Data-7\Attribute_Definitions__product_catalog.txt
Reading file =>  C:\Research-Paper\PAPER-WORK-2024\Spider-Data-7\bank__loan_1.txt
Reading file =>  C:\Research-Paper\PAPER-WORK-2024\Spider-Data-7\Catalogs__product_catalog.txt
Reading file =>  C:\Research-Paper\PAPER-WORK-2024\Spider-Data-7\Catalog_Contents_Additional_Attributes__product_catalog.txt
Reading file =>  C:\Research-Paper\PAPER-WORK-2024\Spider-Data-7\Catalog_Contents__product_catalog.txt
Reading file =>  C:\Research-Paper\PAPER-WORK-2024\Spider-Data-7\Catalog_Structure__product_catalog.txt
Reading file =>  C:\Research-Paper\PAPER-WORK-2024\Spider-Data-7\church__wedding.txt
Reading file =>  C:\Research-Paper\PAPER-WORK-2024\Spider-Data-7\city__farm.txt
Reading file =>  C:\Research-Paper\PAPER-WORK-2024\Spider-Data-7\Claims__insurance_policies.txt
Reading file =>  C:\Research-Paper\PAPER-WORK-2024\Spider-Data

In [7]:
answer

[('/*\nThe farm table contains detailed data about the livestock population and farm resources across multiple years. The table tracks the total number of different types of animals on the farm and distinguishes between various categories such as horses, cattle, oxen, bulls, cows, pigs, and sheep/goats.\n\nFarm_ID\tA unique identifier for each farm entry.\nYear\tThe year for which the data is recorded.\nTotal_Horses\tThe total number of horses on the farm.\nWorking_Horses\tThe number of horses used for working purposes on the farm.\nTotal_Cattle\tThe total number of cattle (oxen, bulls, and cows) on the farm.\nOxen\tThe number of oxen specifically on the farm.\nBulls\tThe number of bulls specifically on the farm.\nCows\tThe number of cows specifically on the farm.\nPigs\tThe total number of pigs on the farm.\nSheep_and_Goats\tThe total number of sheep and goats on the farm.\n*/\n\nCREATE TABLE "farm" (\n"Farm_ID" int,\n"Year" int,\n"Total_Horses" real,\n"Working_Horses" real,\n"Total_C

In [8]:
import math

def calculate_dcg(scores):
    """
    Calculate Discounted Cumulative Gain (DCG) for a given list of scores.
    
    :param scores: List of scores (relevance scores of the documents).
    :return: DCG value.
    """
    dcg = 0.0
    for i, score in enumerate(scores):
        # DCG formula: sum(score / log2(position + 1))
        dcg += score / math.log2(i + 2)  # i + 2 because position starts at 1
    return dcg

# Example usage:
scores = [1.6630921, 1.6599672, 1.2435626]
dcg_value = calculate_dcg(scores)
print(f"DCG for the scores {scores}: {dcg_value:.4f}")

DCG for the scores [1.6630921, 1.6599672, 1.2435626]: 3.3322


In [9]:
def calculate_std_dev(values):
    """
    Calculate the standard deviation of a list of numbers.
    
    :param values: List of numeric values.
    :return: Standard deviation.
    """
    if len(values) == 0:
        raise ValueError("The list of values cannot be empty.")
    
    mean = sum(values) / len(values)
    variance = sum((x - mean) ** 2 for x in values) / len(values)
    std_dev = math.sqrt(variance)
    return std_dev

# Example usage:
scores = [1.6630921, 1.6599672, 1.2435626]
std_dev_value = calculate_std_dev(scores)
print(f"Standard Deviation for the scores {scores}: {std_dev_value:.4f}")

Standard Deviation for the scores [1.6630921, 1.6599672, 1.2435626]: 0.1970


In [10]:
def calculate_average(scores):
    if len(scores) == 0:
        raise ValueError("The list of scores cannot be empty.")
    
    average = sum(scores) / len(scores)
    return average

# Example usage:
scores = [1.6630921, 1.6599672, 1.2435626]
std_dev_value = calculate_average(scores)
print(f"Average for the scores {scores}: {std_dev_value:.4f}")

Average for the scores [1.6630921, 1.6599672, 1.2435626]: 1.5222


In [11]:
rag_table_1_list,rag_table_2_list,rag_table_3_list = [],[],[]
rag_table_1_score_list,rag_table_2_score_list,rag_table_3_score_list = [],[],[]
dcg_score_list = []
std_dev_score_list = []
range_score_list = []

i = 0
for index, row in spider_excel_df.iterrows():
    i = i+1
    print("--------------------- Number : ",i," ---------------------")
    question = row['question']
    answer = similarity_k_search(question,k=3) # RAG result
    
    # Insert Rag Scores
    rag_table_1_score_list.append(answer[0][1])
    rag_table_2_score_list.append(answer[1][1])
    rag_table_3_score_list.append(answer[2][1])
    
    score_list = [answer[0][1],answer[1][1],answer[2][1]]
    
    # Calculate DCG of Score
    dcg_score_list.append(calculate_dcg(score_list))
    
    # Calculate Std Dev of Score
    std_dev_score_list.append(calculate_std_dev(score_list))
    
    # Calculate Range of Score
    range_score_list.append(answer[2][1]-answer[0][1])
    
    # Insert table names (In format <table_name>__<dbid>)
    rag_table_1_list.append(answer[0][2]+"__"+answer[0][3])
    rag_table_2_list.append(answer[1][2]+"__"+answer[1][3])
    rag_table_3_list.append(answer[2][2]+"__"+answer[2][3])    

--------------------- Number :  1  ---------------------
Using cached database...
farm  from db_id :  farm  scored ==> 0.98307145
farm_competition  from db_id :  farm  scored ==> 1.4487716
competition_record  from db_id :  farm  scored ==> 1.4860706
--------------------- Number :  2  ---------------------
Using cached database...
farm  from db_id :  farm  scored ==> 0.7347885
competition_record  from db_id :  farm  scored ==> 1.3012683
farm_competition  from db_id :  farm  scored ==> 1.3720636
--------------------- Number :  3  ---------------------
Using cached database...
farm  from db_id :  farm  scored ==> 0.66637516
competition_record  from db_id :  farm  scored ==> 1.2602258
farm_competition  from db_id :  farm  scored ==> 1.4009414
--------------------- Number :  4  ---------------------
Using cached database...
farm  from db_id :  farm  scored ==> 0.6697946
competition_record  from db_id :  farm  scored ==> 1.0839683
farm_competition  from db_id :  farm  scored ==> 1.2319887
--

farm_competition  from db_id :  farm  scored ==> 0.9149208
competition_record  from db_id :  farm  scored ==> 1.1367664
city  from db_id :  farm  scored ==> 1.2030962
--------------------- Number :  35  ---------------------
Using cached database...
city  from db_id :  farm  scored ==> 1.0431466
market  from db_id :  film_rank  scored ==> 1.3768792
county  from db_id :  election  scored ==> 1.4298772
--------------------- Number :  36  ---------------------
Using cached database...
city  from db_id :  farm  scored ==> 0.96585923
market  from db_id :  film_rank  scored ==> 1.334347
county  from db_id :  election  scored ==> 1.3481404
--------------------- Number :  37  ---------------------
Using cached database...
city  from db_id :  farm  scored ==> 0.99771136
market  from db_id :  film_rank  scored ==> 1.4167142
county  from db_id :  election  scored ==> 1.4511486
--------------------- Number :  38  ---------------------
Using cached database...
city  from db_id :  farm  scored ==> 0

Catalog_Contents  from db_id :  product_catalog  scored ==> 0.99167275
Catalogs  from db_id :  product_catalog  scored ==> 1.0382881
Catalog_Structure  from db_id :  product_catalog  scored ==> 1.04253
--------------------- Number :  65  ---------------------
Using cached database...
Products  from db_id :  manufactory_1  scored ==> 1.4193972
stock  from db_id :  device  scored ==> 1.4312644
Manufacturers  from db_id :  manufactory_1  scored ==> 1.5414037
--------------------- Number :  66  ---------------------
Using cached database...
Catalogs  from db_id :  product_catalog  scored ==> 0.9621742
Catalog_Contents  from db_id :  product_catalog  scored ==> 1.0826404
Catalog_Structure  from db_id :  product_catalog  scored ==> 1.2442646
--------------------- Number :  67  ---------------------
Using cached database...
stock  from db_id :  device  scored ==> 1.3349388
Products  from db_id :  manufactory_1  scored ==> 1.4545382
Catalog_Contents  from db_id :  product_catalog  scored ==> 1

shop  from db_id :  coffee_shop  scored ==> 0.9782169
shop  from db_id :  device  scored ==> 1.2704316
happy_hour  from db_id :  coffee_shop  scored ==> 1.2824607
--------------------- Number :  95  ---------------------
Using cached database...
shop  from db_id :  coffee_shop  scored ==> 0.98008037
shop  from db_id :  device  scored ==> 1.1595544
market  from db_id :  film_rank  scored ==> 1.3188083
--------------------- Number :  96  ---------------------
Using cached database...
happy_hour  from db_id :  coffee_shop  scored ==> 0.8852384
shop  from db_id :  coffee_shop  scored ==> 1.1152354
shop  from db_id :  device  scored ==> 1.2077329
--------------------- Number :  97  ---------------------
Using cached database...
happy_hour  from db_id :  coffee_shop  scored ==> 0.89900845
shop  from db_id :  device  scored ==> 1.1433145
shop  from db_id :  coffee_shop  scored ==> 1.2088175
--------------------- Number :  98  ---------------------
Using cached database...
happy_hour  from db_

climber  from db_id :  climbing  scored ==> 0.9982478
mountain  from db_id :  climbing  scored ==> 1.1336101
swimmer  from db_id :  swimming  scored ==> 1.5240633
--------------------- Number :  129  ---------------------
Using cached database...
climber  from db_id :  climbing  scored ==> 0.8895763
mountain  from db_id :  climbing  scored ==> 1.2467024
swimmer  from db_id :  swimming  scored ==> 1.410741
--------------------- Number :  130  ---------------------
Using cached database...
climber  from db_id :  climbing  scored ==> 0.92939764
mountain  from db_id :  climbing  scored ==> 1.3342324
swimmer  from db_id :  swimming  scored ==> 1.4187438
--------------------- Number :  131  ---------------------
Using cached database...
mountain  from db_id :  climbing  scored ==> 0.93260986
climber  from db_id :  climbing  scored ==> 1.3182058
city  from db_id :  farm  scored ==> 1.5383153
--------------------- Number :  132  ---------------------
Using cached database...
mountain  from db_

...

stock  from db_id :  device  scored ==> 0.8153236
device  from db_id :  device  scored ==> 1.0704806
Manufacturers  from db_id :  manufactory_1  scored ==> 1.2615991
--------------------- Number :  562  ---------------------
Using cached database...
device  from db_id :  device  scored ==> 1.0143322
stock  from db_id :  device  scored ==> 1.0798415
Manufacturers  from db_id :  manufactory_1  scored ==> 1.4926424
--------------------- Number :  563  ---------------------
Using cached database...
stock  from db_id :  device  scored ==> 0.79828763
shop  from db_id :  device  scored ==> 1.1918952
device  from db_id :  device  scored ==> 1.2275124
--------------------- Number :  564  ---------------------
Using cached database...
stock  from db_id :  device  scored ==> 1.0600743
device  from db_id :  device  scored ==> 1.3622881
Products  from db_id :  manufactory_1  scored ==> 1.4548712
--------------------- Number :  565  ---------------------
Using cached database...
stock  from db_id : 

stadium  from db_id :  swimming  scored ==> 0.95199
market  from db_id :  film_rank  scored ==> 1.3853419
farm_competition  from db_id :  farm  scored ==> 1.435114
--------------------- Number :  596  ---------------------
Using cached database...
swimmer  from db_id :  swimming  scored ==> 0.848802
record  from db_id :  swimming  scored ==> 1.1455028
event  from db_id :  swimming  scored ==> 1.3497767
--------------------- Number :  597  ---------------------
Using cached database...
swimmer  from db_id :  swimming  scored ==> 0.8814156
record  from db_id :  swimming  scored ==> 1.2060416
event  from db_id :  swimming  scored ==> 1.2431535
--------------------- Number :  598  ---------------------
Using cached database...
swimmer  from db_id :  swimming  scored ==> 0.8328916
record  from db_id :  swimming  scored ==> 1.1314942
event  from db_id :  swimming  scored ==> 1.1846023
--------------------- Number :  599  ---------------------
Using cached database...
swimmer  from db_id :  s

manager  from db_id :  railway  scored ==> 1.1969286
railway_manage  from db_id :  railway  scored ==> 1.3728132
happy_hour  from db_id :  coffee_shop  scored ==> 1.413054
--------------------- Number :  631  ---------------------
Using cached database...
manager  from db_id :  railway  scored ==> 0.9129024
railway_manage  from db_id :  railway  scored ==> 1.2218106
Manufacturers  from db_id :  manufactory_1  scored ==> 1.3097483
--------------------- Number :  632  ---------------------
Using cached database...
manager  from db_id :  railway  scored ==> 1.1796978
railway_manage  from db_id :  railway  scored ==> 1.2412937
happy_hour  from db_id :  coffee_shop  scored ==> 1.4281197
--------------------- Number :  633  ---------------------
Using cached database...
manager  from db_id :  railway  scored ==> 1.171888
railway_manage  from db_id :  railway  scored ==> 1.3756772
happy_hour  from db_id :  coffee_shop  scored ==> 1.4241774
--------------------- Number :  634  ----------------

manager  from db_id :  railway  scored ==> 1.5626686
Customer_Policies  from db_id :  insurance_policies  scored ==> 1.5973992
railway_manage  from db_id :  railway  scored ==> 1.5991771
--------------------- Number :  664  ---------------------
Using cached database...
SportsInfo  from db_id :  game_1  scored ==> 1.0524088
swimmer  from db_id :  swimming  scored ==> 1.2698103
stadium  from db_id :  swimming  scored ==> 1.309849
--------------------- Number :  665  ---------------------
Using cached database...
SportsInfo  from db_id :  game_1  scored ==> 1.0034714
swimmer  from db_id :  swimming  scored ==> 1.2184832
stadium  from db_id :  swimming  scored ==> 1.2541187
--------------------- Number :  666  ---------------------
Using cached database...
SportsInfo  from db_id :  game_1  scored ==> 0.8030606
Plays_Games  from db_id :  game_1  scored ==> 1.1852248
swimmer  from db_id :  swimming  scored ==> 1.3202487
--------------------- Number :  667  ---------------------
Using cached

Plays_Games  from db_id :  game_1  scored ==> 1.0122062
SportsInfo  from db_id :  game_1  scored ==> 1.1286871
stadium  from db_id :  swimming  scored ==> 1.3958752
--------------------- Number :  698  ---------------------
Using cached database...
Plays_Games  from db_id :  game_1  scored ==> 0.9727267
Video_Games  from db_id :  game_1  scored ==> 1.3007154
SportsInfo  from db_id :  game_1  scored ==> 1.3329563
--------------------- Number :  699  ---------------------
Using cached database...
Plays_Games  from db_id :  game_1  scored ==> 0.8974451
SportsInfo  from db_id :  game_1  scored ==> 1.1617444
Student  from db_id :  game_1  scored ==> 1.30214
--------------------- Number :  700  ---------------------
Using cached database...
Plays_Games  from db_id :  game_1  scored ==> 0.78646624
Student  from db_id :  game_1  scored ==> 1.0802248
SportsInfo  from db_id :  game_1  scored ==> 1.1076672
--------------------- Number :  701  ---------------------
Using cached database...
Plays_G

In [12]:
print(rag_table_1_list[0],rag_table_2_list[0],rag_table_3_list[0])
print(rag_table_1_score_list[0],rag_table_2_score_list[0],rag_table_3_score_list[0])
print(dcg_score_list[0])
print(calculate_dcg([rag_table_1_score_list[0],rag_table_2_score_list[0],rag_table_3_score_list[0]]))

farm__farm farm_competition__farm competition_record__farm
0.98307145 1.4487716 1.4860706
2.6401798689031963
2.6401798689031963


In [13]:
print(len(dcg_score_list))
print(len(std_dev_score_list))
print(len(range_score_list))
print(len(rag_table_1_list))
print(len(rag_table_2_list))
print(len(rag_table_3_list))
print(len(rag_table_1_score_list))
print(len(rag_table_2_score_list))
print(len(rag_table_3_score_list))

719
719
719
719
719
719
719
719
719


In [14]:
spider_excel_df.head()

,db_id,spider_query,question,text2sql_query
0,farm,SELECT count(*) FROM farm,How many farms are there?,"SELECT COUNT(DISTINCT f.Farm_ID) FROM ""farm"" f;"
1,farm,SELECT count(*) FROM farm,Count the number of farms.,"SELECT COUNT(DISTINCT f.""Farm_ID"") FROM ""farm"" f;"
2,farm,SELECT Total_Horses FROM farm ORDER BY Total_H...,List the total number of horses on farms in as...,"SELECT f.""Farm_ID"", f.""Year"", f.""Total_Horses""..."
3,farm,SELECT Total_Horses FROM farm ORDER BY Total_H...,"What is the total horses record for each farm,...","SELECT f.Farm_ID, f.Total_Horses FROM farm f O..."
4,farm,SELECT Hosts FROM farm_competition WHERE Theme...,What are the hosts of competitions whose theme...,SELECT DISTINCT fc.Hosts FROM farm_competition...


In [15]:
spider_excel_df.shape

(719, 4)

In [16]:
spider_excel_df['table_1'] = rag_table_1_list
spider_excel_df['table_2'] = rag_table_2_list
spider_excel_df['table_3'] = rag_table_3_list

spider_excel_df['score_1'] = rag_table_1_score_list
spider_excel_df['score_2'] = rag_table_2_score_list
spider_excel_df['score_3'] = rag_table_3_score_list

spider_excel_df['dcg_score'] = dcg_score_list
spider_excel_df['std_dev_score'] = std_dev_score_list
spider_excel_df['range_score'] = range_score_list

In [17]:
spider_excel_df.shape

(719, 13)

In [18]:
spider_excel_df.head()

,db_id,spider_query,question,text2sql_query,table_1,table_2,table_3,score_1,score_2,score_3,dcg_score,std_dev_score,range_score
0,farm,SELECT count(*) FROM farm,How many farms are there?,"SELECT COUNT(DISTINCT f.Farm_ID) FROM ""farm"" f;",farm__farm,farm_competition__farm,competition_record__farm,0.983071,1.448772,1.486071,2.640180,0.228832,0.502999
1,farm,SELECT count(*) FROM farm,Count the number of farms.,"SELECT COUNT(DISTINCT f.""Farm_ID"") FROM ""farm"" f;",farm__farm,competition_record__farm,farm_competition__farm,0.734788,1.301268,1.372064,2.241829,0.285196,0.637275
2,farm,SELECT Total_Horses FROM farm ORDER BY Total_H...,List the total number of horses on farms in as...,"SELECT f.""Farm_ID"", f.""Year"", f.""Total_Horses""...",farm__farm,competition_record__farm,farm_competition__farm,0.666375,1.260226,1.400941,2.161960,0.318337,0.734566
3,farm,SELECT Total_Horses FROM farm ORDER BY Total_H...,"What is the total horses record for each farm,...","SELECT f.Farm_ID, f.Total_Horses FROM farm f O...",farm__farm,competition_record__farm,farm_competition__farm,0.669795,1.083968,1.231989,1.969697,0.237934,0.562194
4,farm,SELECT Hosts FROM farm_competition WHERE Theme...,What are the hosts of competitions whose theme...,SELECT DISTINCT fc.Hosts FROM farm_competition...,farm_competition__farm,stadium__swimming,competition_record__farm,1.050166,1.398394,1.418680,2.641794,0.169141,0.368514


An increase in the average DCG from System 1 to System 2 generally suggests that System 2 performs better in ranking documents. However, whether System 2 is definitively "better" depends on the context and how you interpret DCG. Here's a breakdown of what an increase in DCG implies and what additional considerations might be necessary:

What an Increased DCG Indicates
- Better Ranking of Relevant Documents:

DCG rewards systems for placing higher relevance scores on top-ranked documents.
An increase implies System 2 is likely ranking more relevant documents higher than System 1.
- Improved Overall Quality:

Higher DCG across multiple queries means System 2 consistently delivers better rankings.

In [20]:
# Calculate Average DCG Score, Std Dev score, Range Score
avg_dcg = calculate_average(dcg_score_list)
avg_std_dev = calculate_average(std_dev_score_list)
avg_range = calculate_average(range_score_list)

print("Average DCG of Score : ",avg_dcg)
print("Average Std dev of Score : ",avg_std_dev)
print("Average Range of Score : ",avg_range)

Average DCG of Score :  2.3983041892067205
Average Std dev of Score :  0.14951154245470852
Average Range of Score :  0.3468805410600007


In [21]:
import pandas as pd

# Save the DataFrame to an Excel file
########################################################################
######### CHANGE OUTPUT FOLDER HERE ####################################
########################################################################
file_path = r"C:\Research-Paper\PAPER-WORK-2024\Outputs\Spider-Data-7-Rag-Score.xlsx"
spider_excel_df.to_excel(file_path, index=False)

print(f"DataFrame saved successfully to {file_path}")

DataFrame saved successfully to C:\Research-Paper\PAPER-WORK-2024\Outputs\Spider-Data-7-Rag-Score.xlsx
